In [ ]:
!pip install torch transformers librosa pandas scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoModel
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report


CSV_PATH = "/content/drive/MyDrive/IIITH_Text/tess_full_metadata.csv"

EMBEDDING_DIR = "/content/drive/MyDrive/IIITH_Voice/TESS_Time_Embeddings_hubert"

TEXT_MODEL = "bert-base-uncased"

BATCH_SIZE = 8

MAX_LEN = 32

EPOCHS = 20

LR = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


df = pd.read_csv(CSV_PATH)

df["speaker"] = df["speaker"].replace({
    "OA": "OAF"
})

df["embedding_folder"] = df["filepath"].apply(
    lambda x: os.path.basename(
        os.path.dirname(x)
    )
)



valid_rows = []

for idx, row in df.iterrows():

    filename = row["filename"]

    folder = row["embedding_folder"]

    base = os.path.splitext(filename)[0]

    emb_path = os.path.join(
        EMBEDDING_DIR,
        folder,
        f"{base}.npy"
    )

    if os.path.exists(emb_path):

        valid_rows.append(idx)

df = df.iloc[
    valid_rows
].reset_index(drop=True)

print("TOTAL VALID:", len(df))



label_encoder = LabelEncoder()

df["label_id"] = label_encoder.fit_transform(
    df["emotion"]
)

NUM_CLASSES = len(label_encoder.classes_)

print(label_encoder.classes_)



splitter = GroupShuffleSplit(
    test_size=0.5,
    n_splits=1,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        df,
        groups=df["speaker"]
    )
)

train_df = df.iloc[train_idx]

val_df = df.iloc[val_idx]

print("\nTRAIN SPEAKERS:")
print(train_df["speaker"].unique())

print("\nVAL SPEAKERS:")
print(val_df["speaker"].unique())



tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL
)



class EmotionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        embedding_dir,
        tokenizer,
        max_len=32
    ):

        self.df = dataframe.reset_index(drop=True)

        self.embedding_dir = embedding_dir

        self.tokenizer = tokenizer

        self.max_len = max_len

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        filename = row["filename"]

        folder = row["embedding_folder"]

        text = row["transcription"]

        label = row["label_id"]

        base = os.path.splitext(
            filename
        )[0]

        emb_path = os.path.join(
            self.embedding_dir,
            folder,
            f"{base}.npy"
        )

        speech_embedding = np.load(
            emb_path
        )

        speech_embedding = torch.tensor(
            speech_embedding,
            dtype=torch.float
        )

        speech_length = speech_embedding.shape[0]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoding[
            "input_ids"
        ].squeeze(0)

        attention_mask = encoding[
            "attention_mask"
        ].squeeze(0)

        return {

            "speech": speech_embedding,

            "speech_length": speech_length,

            "input_ids": input_ids,

            "attention_mask": attention_mask,

            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }



def collate_fn(batch):

    speech = [
        item["speech"]
        for item in batch
    ]

    speech_lengths = torch.tensor([

        item["speech_length"]

        for item in batch
    ])

    speech = pad_sequence(
        speech,
        batch_first=True,
        padding_value=0
    )

    max_len = speech.size(1)

    speech_mask = torch.arange(
        max_len
    ).expand(
        len(speech_lengths),
        max_len
    ) < speech_lengths.unsqueeze(1)

    input_ids = torch.stack([
        item["input_ids"]
        for item in batch
    ])

    attention_mask = torch.stack([
        item["attention_mask"]
        for item in batch
    ])

    labels = torch.stack([
        item["labels"]
        for item in batch
    ])

    return {

        "speech": speech,

        "speech_mask": speech_mask,

        "input_ids": input_ids,

        "attention_mask": attention_mask,

        "labels": labels
    }



train_dataset = EmotionDataset(
    train_df,
    EMBEDDING_DIR,
    tokenizer
)

val_dataset = EmotionDataset(
    val_df,
    EMBEDDING_DIR,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)



class AttentionPooling(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.attention = nn.Linear(
            hidden_dim,
            1
        )

    def forward(
        self,
        x,
        mask=None
    ):

        scores = self.attention(
            x
        ).squeeze(-1)

        if mask is not None:

            scores = scores.masked_fill(
                mask == 0,
                -1e9
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        weights = weights.unsqueeze(-1)

        pooled = torch.sum(
            weights * x,
            dim=1
        )

        return pooled



class MultimodalEmotionModel(nn.Module):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.speech_lstm = nn.LSTM(

            input_size=768,

            hidden_size=128,

            batch_first=True,

            bidirectional=True,

            num_layers=1
        )

        self.speech_attention = AttentionPooling(
            256
        )

        self.speech_fc = nn.Sequential(

            nn.Linear(256, 128),

            nn.ReLU(),

            nn.Dropout(0.5)
        )

        self.text_encoder = AutoModel.from_pretrained(
            TEXT_MODEL
        )

        # FREEZE BERT
        for param in self.text_encoder.parameters():

            param.requires_grad = False

        self.text_fc = nn.Sequential(

            nn.Linear(768, 64),

            nn.ReLU(),

            nn.Dropout(0.5)
        )

        fusion_dim = 128 + 64

        self.classifier = nn.Sequential(

            nn.Linear(
                fusion_dim,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(
        self,
        speech,
        speech_mask,
        input_ids,
        attention_mask
    ):

        speech_out, _ = self.speech_lstm(
            speech
        )

        speech_repr = self.speech_attention(
            speech_out,
            speech_mask
        )

        speech_repr = self.speech_fc(
            speech_repr
        )

        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_repr = torch.mean(
            text_outputs.last_hidden_state,
            dim=1
        )

        text_repr = self.text_fc(
            text_repr
        )


        speech_repr = speech_repr * 0.8

        text_repr = text_repr * 0.2

        fusion = torch.cat([

            speech_repr,

            text_repr

        ], dim=1)

        logits = self.classifier(
            fusion
        )

        return logits


model = MultimodalEmotionModel(
    num_classes=NUM_CLASSES
)

model.to(DEVICE)


criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)


optimizer = torch.optim.AdamW(

    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),

    lr=LR,

    weight_decay=1e-4
)


best_val_acc = 0

patience = 5

counter = 0


for epoch in range(EPOCHS):


    model.train()

    total_loss = 0

    correct = 0

    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for batch in progress_bar:

        speech = batch["speech"].to(DEVICE)

        speech_mask = batch[
            "speech_mask"
        ].to(DEVICE)

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            speech,
            speech_mask,
            input_ids,
            attention_mask
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    train_acc = correct / total

    train_loss = total_loss / len(train_loader)

    model.eval()

    val_correct = 0

    val_total = 0

    val_loss = 0

    all_preds = []

    all_labels = []

    with torch.no_grad():

        for batch in val_loader:

            speech = batch["speech"].to(DEVICE)

            speech_mask = batch[
                "speech_mask"
            ].to(DEVICE)

            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            attention_mask = batch[
                "attention_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            outputs = model(
                speech,
                speech_mask,
                input_ids,
                attention_mask
            )

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                preds == labels
            ).sum().item()

            val_total += labels.size(0)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    val_acc = val_correct / val_total

    val_loss /= len(val_loader)

    print("\n")

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.4f}"
    )

    print("\nClassification Report:\n")

    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=label_encoder.classes_
        )
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        counter = 0

        torch.save(

            {

                "model_state_dict":
                    model.state_dict(),

                "label_encoder_classes":
                    label_encoder.classes_

            },

            "/content/drive/MyDrive/IIITH_Text/best_multimodal_model.pt"
        )

        print("\nBEST MODEL SAVED")

    else:

        counter += 1

        print(
            f"\nEarlyStopping Counter: {counter}/{patience}"
        )

        if counter >= patience:

            print("\nEARLY STOPPING")

            break


print(
    f"\nBest Validation Accuracy: {best_val_acc:.4f}"
)

TOTAL VALID: 2800
['angry' 'disgust' 'fear' 'happy' 'neutral' 'ps' 'sad']

TRAIN SPEAKERS:
['OAF']

VAL SPEAKERS:
['YAF']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/20: 100%|██████████| 175/175 [00:11<00:00, 15.57it/s]




Epoch 1/20
Train Loss: 1.9396
Train Acc: 0.1893
Val Loss: 1.9305
Val Acc: 0.2221

Classification Report:

              precision    recall  f1-score   support

       angry       0.06      0.05      0.06       200
     disgust       0.00      0.00      0.00       200
        fear       0.00      0.00      0.00       200
       happy       0.00      0.00      0.00       200
     neutral       0.00      0.00      0.00       200
          ps       0.20      1.00      0.33       200
         sad       0.42      0.51      0.46       200

    accuracy                           0.22      1400
   macro avg       0.10      0.22      0.12      1400
weighted avg       0.10      0.22      0.12      1400



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...



Epoch 11/20
Train Loss: 0.5182
Train Acc: 0.9986
Val Loss: 1.4290
Val Acc: 0.5607

Classification Report:

              precision    recall  f1-score   support

       angry       0.16      0.18      0.17       200
     disgust       0.58      0.98      0.73       200
        fear       1.00      0.80      0.89       200
       happy       0.13      0.14      0.13       200
     neutral       0.98      1.00      0.99       200
          ps       0.00      0.00      0.00       200
         sad       0.68      0.81      0.74       200

    accuracy                           0.56      1400
   macro avg       0.50      0.56      0.52      1400
weighted avg       0.50      0.56      0.52      1400


EarlyStopping Counter: 3/5


Epoch 12/20: 100%|██████████| 175/175 [00:10<00:00, 16.11it/s]




Epoch 12/20
Train Loss: 0.5127
Train Acc: 0.9979
Val Loss: 1.3966
Val Acc: 0.6057

Classification Report:

              precision    recall  f1-score   support

       angry       0.10      0.07      0.08       200
     disgust       0.89      0.93      0.90       200
        fear       0.97      0.95      0.96       200
       happy       0.23      0.32      0.27       200
     neutral       0.96      1.00      0.98       200
          ps       0.20      0.01      0.01       200
         sad       0.52      0.97      0.68       200

    accuracy                           0.61      1400
   macro avg       0.55      0.61      0.56      1400
weighted avg       0.55      0.61      0.56      1400


EarlyStopping Counter: 4/5


Epoch 13/20: 100%|██████████| 175/175 [00:10<00:00, 16.37it/s]




Epoch 13/20
Train Loss: 0.5101
Train Acc: 1.0000
Val Loss: 1.3876
Val Acc: 0.5793

Classification Report:

              precision    recall  f1-score   support

       angry       0.20      0.22      0.21       200
     disgust       0.88      0.84      0.86       200
        fear       0.93      0.96      0.95       200
       happy       0.12      0.14      0.13       200
     neutral       0.93      1.00      0.96       200
          ps       0.05      0.01      0.01       200
         sad       0.57      0.88      0.69       200

    accuracy                           0.58      1400
   macro avg       0.53      0.58      0.54      1400
weighted avg       0.53      0.58      0.54      1400


EarlyStopping Counter: 5/5

EARLY STOPPING

Best Validation Accuracy: 0.6186
